In [ ]:
import os
from PIL import Image
import torch
import torch.nn.functional as F
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
from huggingface_hub import hf_hub_download
from tqdm import tqdm

# Setup
device = 'cpu'
model = InceptionResnetV1(pretrained=None, classify=False, dropout_prob=0.6)
model.logits = torch.nn.Linear(512, 8631)
model_path = hf_hub_download(repo_id="py-feat/facenet", filename="facenet_20180402_114759_vggface2.pth")
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval().to(device)

# Image preprocessing
transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

def get_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        return model(tensor)

# Paths
original_dir = "ADD DIR PATH"
anonymized_dir = "ADD DIR PATH"

# Loop over all files
similarities = []
file_names = sorted(os.listdir(original_dir))

for file_name in tqdm(file_names, desc="Computing cosine similarities"):
    orig_path = os.path.join(original_dir, file_name)
    anon_path = os.path.join(anonymized_dir, file_name.replace("original", "anon"))

    if not os.path.exists(anon_path):
        print(f"Missing anonymized image for {file_name}, skipping...")
        continue

    try:
        emb_orig = get_embedding(orig_path)
        emb_anon = get_embedding(anon_path)

        cos_sim = F.cosine_similarity(emb_orig, emb_anon).item()
        similarities.append(cos_sim)
    except Exception as e:
        print(f"Error with {file_name}: {e}")

# Report
if similarities:
    avg_sim = sum(similarities) / len(similarities)
    print(f"\nAverage Cosine Similarity: {avg_sim:.4f}")
else:
    print("No valid image pairs found.")


Computing cosine similarities: 100%|██████████| 40/40 [00:02<00:00, 14.07it/s]


Average Cosine Similarity: 0.6083


In [3]:
print(embedding)

tensor([[-8.4878e-03, -5.5937e-02, -5.8744e-02,  1.0391e-01,  6.3695e-03,
          4.3282e-02, -1.4224e-02, -1.3788e-02, -2.5441e-02, -2.7659e-02,
         -4.2580e-03,  2.9060e-03,  1.7326e-02, -6.6898e-02,  1.2054e-02,
         -5.5358e-02,  6.0916e-03,  2.9813e-02,  1.8437e-02, -5.3141e-02,
         -3.9200e-02,  3.4614e-02,  6.4046e-02,  6.0069e-02, -2.5126e-02,
          1.2040e-01,  6.2212e-02, -1.5226e-02, -1.3626e-02,  2.6528e-02,
         -4.7971e-02,  5.2979e-02, -7.6986e-02, -2.0655e-02, -3.0564e-02,
          5.0668e-02, -1.9195e-02,  2.3307e-02, -1.1356e-01,  4.5572e-02,
          4.1537e-03,  2.1528e-02, -9.6533e-03, -4.3194e-02,  2.3893e-02,
          2.5486e-03,  7.4065e-02,  4.9392e-02, -4.7005e-02, -3.9708e-02,
         -3.8045e-03,  6.9283e-02,  2.4341e-02,  2.7427e-02, -4.5622e-02,
          6.4015e-02,  3.4448e-02,  5.8420e-02,  7.7352e-02,  6.4635e-03,
          1.7948e-02, -8.1313e-04,  1.9729e-02, -8.7169e-02,  5.8013e-02,
          9.8580e-02, -3.1672e-02, -2.